# MLP from Scratch using NumPy
### Build a Multi-Layer Perceptron without TensorFlow or PyTorch

**Goal:** Implement a small neural network from scratch and train it to classify handwritten digits from the scikit-learn digits dataset.

**What you'll learn**
- How a feedforward neural network works
- Forward propagation
- ReLU and softmax activation functions
- Cross-entropy loss
- Backpropagation and gradient descent
- Evaluating a trained classifier

> This notebook uses only NumPy for the neural-network implementation. Scikit-learn is used only to load the dataset and split it into training and test sets.

## 1. Import libraries

NumPy performs the neural-network calculations. Matplotlib is used to visualize training progress and predictions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

np.random.seed(42)
print("NumPy version:", np.__version__)

## 2. Load and prepare the dataset

The digits dataset contains 1,797 small grayscale images of handwritten digits (0–9). Each image is 8×8 pixels, so it becomes a vector of 64 input features.

We split the data into training and test sets. The scaler is fitted on the training data only to avoid data leakage.

In [ ]:
digits = load_digits()
X = digits.data.astype(np.float64) / 16.0  # Pixel values scaled approximately to [0, 1]
y = digits.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Full dataset:", X.shape)
print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("Classes:", np.unique(y))

## 3. Explore some handwritten digits

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for ax, image, label in zip(axes.ravel(), digits.images[:10], digits.target[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Define the MLP from scratch

Network architecture:

**64 inputs → 32 hidden neurons (ReLU) → 10 output neurons (Softmax)**

- **Forward propagation:** calculates predictions from the input.
- **Softmax:** converts output scores into class probabilities.
- **Cross-entropy loss:** measures how far predictions are from the correct labels.
- **Backpropagation:** calculates gradients of the loss with respect to weights and biases.
- **Gradient descent:** updates weights and biases to reduce loss.

All neural-network operations below are implemented with NumPy.

In [ ]:
class MLP:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.1):
        self.learning_rate = learning_rate

        # He initialization helps keep ReLU activations at a useful scale.
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))

    def relu(self, z):
        return np.maximum(0, z)

    def softmax(self, z):
        # Subtract row maximum for numerical stability.
        z = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.probs = self.softmax(self.z2)
        return self.probs

    def compute_loss(self, probs, y):
        n = y.shape[0]
        eps = 1e-12
        return -np.mean(np.log(probs[np.arange(n), y] + eps))

    def backward(self, X, y):
        n = X.shape[0]

        # Gradient of softmax + cross-entropy loss.
        dz2 = self.probs.copy()
        dz2[np.arange(n), y] -= 1
        dz2 /= n

        dW2 = self.a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * (self.z1 > 0)  # Derivative of ReLU
        dW1 = X.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)

        # Gradient-descent parameter updates.
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1

    def predict_proba(self, X):
        return self.forward(X)

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

## 5. Train the neural network

We train for multiple epochs. Each epoch uses mini-batches, which means the model updates its parameters after seeing a small group of training examples.

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val,
                epochs=100, batch_size=32):
    history = {"loss": [], "train_accuracy": [], "val_accuracy": []}
    n = X_train.shape[0]

    for epoch in range(epochs):
        indices = np.random.permutation(n)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        for start in range(0, n, batch_size):
            end = start + batch_size
            X_batch = X_shuffled[start:end]
            y_batch = y_shuffled[start:end]

            model.forward(X_batch)
            model.backward(X_batch, y_batch)

        # Measure loss and accuracy after the epoch.
        train_probs = model.predict_proba(X_train)
        train_loss = model.compute_loss(train_probs, y_train)
        train_acc = np.mean(np.argmax(train_probs, axis=1) == y_train)
        val_pred = model.predict(X_val)
        val_acc = np.mean(val_pred == y_val)

        history["loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                f"Epoch {epoch+1:3d}/{epochs} | "
                f"Loss: {train_loss:.4f} | "
                f"Train accuracy: {train_acc:.3f} | "
                f"Test accuracy: {val_acc:.3f}"
            )

    return history

model = MLP(input_size=64, hidden_size=32, output_size=10, learning_rate=0.1)
history = train_model(model, X_train, y_train, X_test, y_test, epochs=100, batch_size=32)

## 6. Visualize training performance

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history["loss"])
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")

plt.subplot(1, 2, 2)
plt.plot(history["train_accuracy"], label="Train accuracy")
plt.plot(history["val_accuracy"], label="Test accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## 7. Evaluate on the test set

Accuracy tells us the proportion of correctly classified images. The classification report also shows precision, recall and F1-score for each digit.

In [ ]:
y_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print(f"Final test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=digits.target_names)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.show()

## 8. Inspect sample predictions

Green titles indicate correct predictions; red titles indicate incorrect predictions.

In [ ]:
rng = np.random.default_rng(7)
sample_indices = rng.choice(len(X_test), size=10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.ravel(), sample_indices):
    image = X_test[idx].reshape(8, 8)
    true_label = y_test[idx]
    predicted_label = y_pred[idx]
    ax.imshow(image, cmap="gray")
    ax.set_title(
        f"True: {true_label} | Pred: {predicted_label}",
        color="green" if true_label == predicted_label else "red"
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Try a prediction yourself

Change `sample_index` to inspect a different test image. The index must be between `0` and `len(X_test) - 1`.

In [ ]:
sample_index = 0  # Change this number to try another test image

sample = X_test[sample_index:sample_index + 1]
prediction = model.predict(sample)[0]
probabilities = model.predict_proba(sample)[0]
actual = y_test[sample_index]

plt.figure(figsize=(3, 3))
plt.imshow(sample.reshape(8, 8), cmap="gray")
plt.title(f"Actual: {actual} | Predicted: {prediction}")
plt.axis("off")
plt.show()

print("Predicted digit:", prediction)
print("Actual digit:", actual)
print("Confidence for predicted class:", f"{probabilities[prediction] * 100:.2f}%")
print("Probabilities for digits 0–9:")
print(np.round(probabilities, 3))

## 10. How backpropagation works (summary)

1. **Forward pass:** Input data passes through the hidden layer and output layer.
2. **Loss calculation:** Cross-entropy measures the prediction error.
3. **Output gradients:** Softmax and cross-entropy give the output-layer gradient.
4. **Hidden-layer gradients:** The error is propagated backward through the second layer and ReLU.
5. **Parameter update:** Gradient descent adjusts weights and biases to reduce loss.
6. **Repeat:** Training continues over batches and epochs.

## Conclusion

This notebook demonstrates the core mechanics of a Multi-Layer Perceptron using NumPy. It trains on handwritten digit images and evaluates the model on held-out data.

**Possible extensions**
- Add another hidden layer
- Compare ReLU with sigmoid or tanh
- Try different learning rates and batch sizes
- Add L2 regularization
- Implement momentum-based gradient descent

**Important:** Results can vary slightly with changes to the random seed, network size, learning rate, and number of epochs.